In [10]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import datetime

# Cấu hình giao diện đồ thị
sns.set(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# path="/home/slow_data/Air_Quality/filtered_envisoft_air_quality_weather_data.csv"
path="/home/slow_data/Air_Quality/IQAir_air_quality.csv"

In [12]:
df_aqi = pd.read_csv(path)
df_aqi.head()

,timestamp,station_name,longitude,latitude,aqi,WHO_exposure,PM2.5 (µg/m³),PM10 (µg/m³),O3 (µg/m³),NO2 (µg/m³),SO2 (µg/m³),CO (µg/m³),condition,temperature (°),humidity (%),pressure,wind_speed (km/h),wind_direction
0,2025-04-17 01:00:00,Hà Nội: Đại Học Bách Khoa cổng Parabol đường ...,105.8418,21.005200,167,15.6,78.1,215.0,62.8,9.6,8.2,NaN,Nhiều mây,23,86,1007,13.2,135
1,2025-04-17 01:00:00,Hà Nội: Công viên hồ điều hòa Nhân Chính Khuấ...,105.7947,21.003100,154,12.1,60.3,204.5,10.2,2.5,6.5,2.0,Nhiều mây,23,86,1007,13.4,136
2,2025-04-17 01:00:00,Minh Khai - Bắc Từ Liêm,105.7400,21.050000,132,5.2,26.2,218.7,21.0,NaN,0.1,1.4,Mưa,23,84,1007,12.3,132
3,2025-04-17 01:00:00,Vũng Tàu: Ngã tư Giếng nước - Tp.Vũng Tàu (KK),107.0844,10.367976,83,5.2,26.2,66.6,73.5,3.9,6.2,0.1,Nhiều mây,27,83,1010,19.0,103
4,2025-04-17 01:00:00,Hải Dương: UBND TP. Hải Dương - 106 Đường Trần...,106.3357,20.938100,144,10.6,53.0,147.4,48.2,1.0,1.3,2.2,Nhiều mây,21,88,1007,11.7,119


In [ ]:
# QB_filter = df_aqi[df_aqi['Name'] == 'Quảng Bình: KKT Hòn La (KK)']
# QB_filter.head()

# Extract AOD for stations

In [14]:
import rasterio
import glob
from  pathlib import Path

In [15]:
# unique_stations = df_aqi['Name'].unique()
# station_df = df_aqi.groupby('Name')[['ID','Latitude', 'Longitude']].first().reset_index()

unique_stations = df_aqi['station_name'].unique()
station_df = df_aqi.groupby('station_name')[['latitude', 'longitude']].first().reset_index()

In [16]:
station_df

,station_name,latitude,longitude
0,"Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến",21.003100,105.794700
1,FPT,10.841600,106.809100
2,HCM - FPT Thuduc,10.841600,106.809100
3,Hà Nội: Chi cục BVMT (KK),21.015250,105.800130
4,Hà Nội: Công viên hồ điều hòa Nhân Chính Khuấ...,21.003100,105.794700
5,Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm...,21.035584,105.852771
6,Hà Nội: Đại Học Bách Khoa cổng Parabol đường ...,21.005200,105.841800
7,Hải Dương: UBND TP. Hải Dương - 106 Đường Trần...,20.938100,106.335700
8,IQAir Ha Noi,21.067900,105.826200
9,IQAir Vietnam - Saigon Pearl,10.790500,106.718700


In [ ]:
aod_dir = "/home/slow_data/Air_Quality/AOD/L2_AOD"
OUTPUT_DIR = "/home/slow_data/Air_Quality/AOD/station_aod/IQAir_stations"

aod_path = Path(aod_dir)

In [ ]:
import os
import glob
import rasterio
import pandas as pd
import numpy as np

# Setup output directory to keep things organized
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Define the pattern
pattern = os.path.join(aod_dir, r'2025??/*/*/aod_vietnam_NC_H??_*_L2ARP031_FLDK.*.tif')
print("Search pattern:", pattern)
files = glob.glob(pattern)
files.sort() # Good practice to process in order
print(f"Found {len(files)} files.")

# Define the columns we want in the final CSVs
band_names = ['AOT', 'Uncertainty', 'AE', 'QA_flag', 'SSA', 'RF']
columns = ['timestamp'] + band_names

for aod_file in files:
    print("Processing file: ", aod_file)
    
    try:
        # 1. Parse Timestamp
        filename = os.path.basename(aod_file)
        parts = filename.split("_")
        timestamp = parts[4] + "_" + parts[5]

        # 2. Read Image Data
        with rasterio.open(aod_file) as src:
            # Read all required bands at once to keep memory handy
            # Bands are 1-indexed in rasterio
            bands_data = [src.read(i) for i in range(1, 7)] 
            
            # 3. Iterate through each station
            for _, row in station_df.iterrows():
                # station_id = str(row['ID'])
                # lon, lat = row["Longitude"], row["Latitude"]

                station_id = str(row['station_name'])
                lon, lat = row["longitude"], row["latitude"]
                
                # Define the output file path for this specific station
                station_csv_path = os.path.join(OUTPUT_DIR, f"{station_id}.csv")
                
                # timestamp = pd.to_datetime(timestamp, format='%Y%m%d_%H%M')
                extracted_values: dict[str, object] = {'timestamp': timestamp}
                
                try:
                    # Get pixel coordinates
                    rowcol = src.index(lon, lat)
                    
                    # Extract data for all bands
                    for i, name in enumerate(band_names):
                        # bands_data is 0-indexed list, so i=0 corresponds to Band 1
                        val = bands_data[i][rowcol[0], rowcol[1]]
                        extracted_values[name] = val
                        
                except Exception as e:
                    # If coordinate is out of bounds or error occurs, ensure band keys remain NaN
                    for name in band_names:
                        extracted_values[name] = np.nan

                # 4. Save to CSV (Append Mode)
                new_df = pd.DataFrame([extracted_values])
                
                # Check if file exists to determine if we need to write the header
                file_exists = os.path.isfile(station_csv_path)
                
                # mode='a' appends to the file instead of overwriting
                new_df.to_csv(station_csv_path, mode='a', header=not file_exists, index=False)

        print(f"✅ Processed timestamp {timestamp}")

    except Exception as e:
        print(f"❌ Error processing file {aod_file}: {e}")

print("All processing complete.")

Search pattern: /home/slow_data/Air_Quality/AOD/L2_AOD/2025??/*/*/aod_vietnam_NC_H??_*_L2ARP031_FLDK.*.tif
Found 47524 files.
Processing file:  /home/slow_data/Air_Quality/AOD/L2_AOD/202501/01/00/aod_vietnam_NC_H09_20250101_0000_L2ARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0000
Processing file:  /home/slow_data/Air_Quality/AOD/L2_AOD/202501/01/00/aod_vietnam_NC_H09_20250101_0010_L2ARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0010
Processing file:  /home/slow_data/Air_Quality/AOD/L2_AOD/202501/01/00/aod_vietnam_NC_H09_20250101_0020_L2ARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0020
Processing file:  /home/slow_data/Air_Quality/AOD/L2_AOD/202501/01/00/aod_vietnam_NC_H09_20250101_0030_L2ARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0030
Processing file:  /home/slow_data/Air_Quality/AOD/L2_AOD/202501/01/00/aod_vietnam_NC_H09_20250101_0040_L2ARP031_FLDK.02401_02401.tif
✅ Processed timestamp 20250101_0040
Processing file:  /home/slow_

In [19]:
df_aod = pd.read_csv("/home/slow_data/Air_Quality/AOD/station_aod/31390903576425084107499649578.csv", parse_dates=[0])

In [20]:
df_aod.head()

,timestamp,AOT,Uncertainty,AE,QA_flag,SSA,RF
0,2025-01-01 07:00:00,NaN,NaN,NaN,5112.0,NaN,NaN
1,2025-01-01 07:10:00,NaN,NaN,NaN,5112.0,NaN,NaN
2,2025-01-01 07:20:00,NaN,NaN,NaN,5112.0,NaN,NaN
3,2025-01-01 07:30:00,NaN,NaN,NaN,5112.0,NaN,NaN
4,2025-01-01 07:40:00,NaN,NaN,NaN,5112.0,NaN,NaN


In [21]:
df_aod.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47524 entries, 0 to 47523
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   timestamp    47524 non-null  datetime64[ns]
 1   AOT          1554 non-null   float64       
 2   Uncertainty  1549 non-null   float64       
 3   AE           1549 non-null   float64       
 4   QA_flag      47524 non-null  float64       
 5   SSA          1549 non-null   float64       
 6   RF           1549 non-null   float64       
dtypes: datetime64[ns](1), float64(6)
memory usage: 2.5 MB


In [22]:
import pandas as pd
import glob
import os

# Get all CSV files
csv_files = glob.glob(os.path.join(OUTPUT_DIR, "*.csv"))
print(f"Found {len(csv_files)} files to update.")

for file_path in csv_files:
    try:
        # 1. Read the CSV
        df = pd.read_csv(file_path)

        df['timestamp'] = pd.to_datetime(df['timestamp'], format='%Y%m%d_%H%M')
        
        # 3. Add 7 hours to convert UTC -> GMT+7
        df['timestamp'] = df['timestamp'] + pd.Timedelta(hours=7)

        df['timestamp'] = df['timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S')
        
        # 5. Overwrite the file
        df.to_csv(file_path, index=False)
        
        print(f"Updated {os.path.basename(file_path)}")
        
    except Exception as e:
        print(f"❌ Error updating {file_path}: {e}")

print("✅ Timezone conversion complete.")

Found 19 files to update.
Updated Quảng Bình: Khu kinh tế Hòn La (KK).csv
Updated Hà Nội: Chi cục BVMT (KK).csv
Updated Công viên hồ điều hòa Nhân Chính, Khuất Duy Tiến.csv
Updated Trà Vinh: Tp. Trà Vinh (KK).csv
Updated Hà Nội: Công viên hồ điều hòa Nhân Chính  Khuất Duy Tiến (KK).csv
Updated Minh Khai - Bắc Từ Liêm.csv
Updated Hà Nội: Đại Học Bách Khoa cổng Parabol đường Giải Phóng (KK).csv
Updated HCM - FPT Thuduc.csv
Updated Hà Nội: TT giao lưu văn hóa phố cổ - Hoàn Kiếm - Trạm cảm biến(KK).csv
Updated IQAir Ha Noi.csv
Updated Thái nguyên: Đường Hùng Vương - Tp Thái Nguyên (KK).csv
Updated ĐH Bách Khoa - cổng Parabol đường Giải Phóng.csv
Updated IQAir Vietnam - Saigon Pearl.csv
Updated Vũng Tàu: Ngã tư Giếng nước - Tp.Vũng Tàu (KK).csv
Updated Office IQAir Ha Noi.csv
Updated Thừa Thiên Huế: 83 đường Hùng Vương (KK).csv
Updated FPT.csv
Updated Hải Dương: UBND TP. Hải Dương - 106 Đường Trần Hưng Đạo (KK).csv
Updated Thái Bình: Cầu Thái Bình - Đ. Trần Thái Tông - P. Bồ Xuyên - TP Thá